# Item 99: Consider `memoryview` and `bytearray` for Zero-Copy

Interactions with `bytes`

## Notes

-   Python requires extra effort to parallelise CPU-bound computation
    (See [Item 79](../../Chapter_09/Item_079/item_079.qmd) and [Item
    94](../Item_094/item_094.qmd))
-   But, can support high-throughput parallel I/O (See [Item
    68](../../Chapter_09/Item_068/item_068.qmd) and [Item
    75](../../Chapter_09/Item_075/item_075.qmd))
-   However, understanding the tools available and how to use them
    *without* leading to slow code can require some skill
-   For example, consider a media-streaming server
    -   Users don’t need to download a video in advance
    -   Users can move forward or backward within a video
-   We might have functions to implement this by converting a time-code
    to a index and returning the associated chunk of data

In [1]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    # simulate by returning random data
    return os.urandom(size)


video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode)
size = (8**2)
video_data = request_chunk(video_id, byte_offset, size)

print(f"{video_id=}, {timecode=}, {byte_offset=}, {video_data=}")

video_id=1, timecode='01:09:14:28', byte_offset=0, video_data=b'WM%\xac\xad\xbc\x9b^\xd3\x9d\x18\x85\xe0\xddU\xd7\xa8\xe7\x1f\x0c\x940^\x1fG\xb8\xaf{\x18\x91\xa0J\x9fW\x04\xd4\xe5\xa5\x12A\xbe\x15\xb7\x1cy,kN\xaa\xda\xde\xc4R8lx\xf3\xc0~\x91:]\xfc\xef'

-   How do we now implement the server-side handler that receives
    `request_chunk`
    -   Must then return the associated video data chunk
-   First we assume that the program is driven by an `asyncio` process
    (See [Item 76](../../Chapter_09/Item_076/item_076.qmd))
    -   Now want to focus on how to handle extracting the chunk
    -   Assume video is cached memory
    -   Extracted then sent over a socket back to a client

In [2]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    return video_data[byte_offset : byte_offset + size]

# Adding in the handling

# simulate a socket connection
class NullSocket:
    def __init__(self):
        self.handle = open(os.devnull, "wb")

    def send(self, data):
        self.handle.write(data)

socket = NullSocket() # represents client socket connection
size = (8 ** 2) # Requested chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video_id

video_id = 1
timecode = "01:09:14:28"

byte_offset = timecode_to_index(video_id, timecode)
chunk = request_chunk(video_id, byte_offset, size)
socket.send(chunk)

print(f"Sent {chunk=} over socket")

Sent chunk=b'\xd8\xd6\x1c\xe3f1\xf9\x96\xcfl\xaf@(\x98G\xa5T\xd2\x05\x8a\xf8\xaep\x1f\xde\xd0\xe1\xbe7\x16\x05\x08X\x93F\xdfN\x01\xb7+\xc2\x0f\xfd\xba\x1d\r\xde\x05\xf1\xff\x1f\xcb<l\xfcy\xce\x12s\xd0\xfce\xb2\x88' over socket

-   Latency and throughput determined by two factors
    1.  How long to slice the chunk from `video_data`
    2.  How long to transmit over a socket
-   Focusing just on point 1, we can microbenchmark how long fetching a
    chunk takes.
    -   We’ll also exclude the function call wrapper
    -   Here we’ll set the size to $20$ MB.

In [3]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
byte_offset = 0

def run_test():
    chunk = video_data[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.001087789 seconds

-   This takes about $5$ milliseconds
-   Theoretical server maximum throughput is thus, limited by video
    extraction speed as

$$
\begin{align}
    \frac{20 \text{ MB}}{5 \text{ ms}} &= 4 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Server also limited to,

$$
\begin{align}
    \frac{1 \text{ CPU=second}}{5 \text{ ms}} &= 200 \text{ clients in parallel}
\end{align}
$$

-   But we already know that `asyncio` should be able to scale up to
    tens of thousands of simultaneous connections
-   The slowdown is because as discussed slices create copies
    -   Copying consumes CPU time
-   Instead we can use `memoryview`
    -   A built-in type for handling the CPython `buffer` protocol
        -   Low-level C API allowing Python runtime and C extensions
            (See [Item 96](../Item_096/item_096.qmd)) to access
            underlying data buffers
            -   Can then treat them as `bytes` instances
        -   Since Python 3.12 the buffer protocol is also emulatable in
            python
-   `memoryview` can be sliced to create a new `memoryview` without a
    copy

In [4]:
data = b"shave and a haircut, two bits"
view = memoryview(data)
chunk = view[12:19]

print(chunk)
print("Size:            ", chunk.nbytes)
print("Data in view:    ", chunk.tobytes())
print("Underlying data: ", chunk.obj)

Size:             7
Data in view:     b'haircut'
Underlying data:  b'shave and a haircut, two bits'

-   These *zero-copy* operations can significantly speed-up code that
    heavily processes memory, e.g.
    1.  I/O-bound access
    2.  Heavy numerical mathematics (e.g. Numpy)
-   Using `memoryview` as a drop-in replacement for our video serving
    service

In [5]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
video_view = memoryview(video_data)
byte_offset = 0

def run_test():
    chunk = video_view[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000000225 seconds

-   This should run in a several hundred nanoseconds
-   So an order of magnitude faster than the `bytes` slicing technique
-   Our new theoretical maximum throughput is then

$$
\begin{align}
\frac{20 \text{ MB}}{250 \text{ ns}} &= 80 \text{ TB}\text{s}^{-1}
\end{align}
$$

-   Or in terms of parallel clients

$$
\begin{align}
\frac{1 \text{ CPU-second}}{250 \text{ ns}} &= 4 \times 10^{9}
\end{align}
$$

-   So four million clients. Now the program should be bound by the
    socket performance rather than CPU constraints.

-   Now consider a reversed process

    -   Users must submit live video streams that are then broadcast out
        to viewers

-   We need to store incoming video data

    -   Cache it for clients to read from

In [6]:
import os

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder

# socket connection from client


size = (4 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]

video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode) # Incoming buffer position
video_view = memoryview(video_cache)


class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
chunk = socket.recv(size)
before = video_view[:byte_offset]
after = video_view[byte_offset + size:]

new_cache = b"".join([before, chunk, after])

print(f"Updated the cache: {new_cache=}")

Updated the cache: new_cache=b'(e\xf1C\\\xe7\x02c\x1a\xc7u\x86\xaa\x9bT\x1c\xee\xad`\xf7\xd8Fj\x02\xb2\xe3E-+:\xa5\x9bO\xdb\xa1\x9a O\xad\x0f\xa3\xd9\xc7\xce\x0fA\xba\xaev\xd0\xed\xa9L\xef\x9d\xecwh\xc5\xa7\x91\xf3\xe5\n\xa8\xafB\x06/,\xa1\xc8\xd2\xea\xed\xbc\xbc\xbc7\x0eUU\x80\n5\xb1\x16JD\x9a\x1f#\x80\x1chO\x8d\xe0\x7f2\xec-V-\x04\x0f9\xa5\xc2\n\xd9\x0f\x16\xfa\xfe\xa4\xd1\x9d\x80\xc3r\x07a#\x93\x16\x17\x8c\x85\xfd\x86\x95{\xeep^\xbe]\xdb\xb6\xce\xa6s\xe3\xe2o\xe2\xf4\xd0\x81\xd41\xed\'\x86\xb2s\xab\xd0\xb4\xa4\xec"\xc5(\x95\x8bZ@)Y$\xd2\xec\xf3\x1d\xfc\x8b\x18\xcfP\x13\x1f\xbd\xcaQc\xfc\xe8\xbf6o\xc7n\x00UC\xb3:^_\x7f\xbd\x8eq\x03\x82\x1c\x80\xc6s%g\x04.\xa4\xa3\xd2\xeb\xd8\xcc\x03;\x8e\x91+tt\x8eC\xe96$\xb1\xf65\xb9+\xb6p\xa4\x99\x07hS\xfcx\x01\xa5\xdc\x86\nY>G[\xa1\xbd^(\xbc]&\x1c.Sa\xad4\x02\xf7\\\xa66\xac\xa4\xe0\x08\xe74b\xab\x07!\xe6\xc7\xc7\x96\x9bG\xb4y\xb1H\x91\x9f\xcd\xf8\x9d\x8c;\t\xb0\xfe@\xd9G\xc9\x90\xd5\xffWL,\xaf \xcc)}\xf3'

-   `socket.recv` returns a `bytes` instance
    -   Splice this into the existing cache
    -   Insert at the current `byte_offset` via slicing and `bytes.join`
-   Now need to profile the timing

In [7]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
size = (1024 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)
byte_offset = 1234 # pick arbitrary point in the middle

def run_test():
    chunk = socket.recv(size)
    before = video_view[:byte_offset]
    after = video_view[byte_offset + size : ]
    new_cache = b"".join([before, chunk, after])

result = (timeit.timeit(stmt="run_test()", globals=globals(), number=100,) / 100)

print(f"{result:0.9f} seconds")

0.001076150 seconds

-   This takes about three milliseconds to receive $1$ MB and update the
    cache.
-   Maximum throughput to receive is then

$$
\begin{align}
\frac{1 \text{ MB}}{ 3 \text{ ms}} &\approx 330 \text{ MB}\text{s}^{-1}
\end{align}
$$

-   Means we are limited to about $300$ simultaneously streaming clients
-   Can use `bytearray` instead of `memoryview`
    -   `bytes` are immutable like strings

In [8]:
some_bytes = b"hello"
some_bytes[0] = 0x79

-   `bytearray` is effectively a mutable version of `bytes`
    -   Can overwrite indices
-   `bytearray` values are integers rather than bytes

In [9]:
array = bytearray(b"hello")
array[0] = 0x79
print(array)

bytearray(b'yello')

-   Can still wrap a `bytearray` in a `memoryview` to avoid extra copies
    -   Then can slice the `memoryview` and modify to overwrite the
        underlying `bytearray`

In [10]:
array = bytearray(b"row, row, row your boat")
view = memoryview(array)
write_view = view[3:13]
write_view[:] = b"-10 bytes-"
print(array)

bytearray(b'row-10 bytes- your boat')

-   Library methods in Python user the buffer protocol for fast data
    receipt or reading, e.g.
    1.  `socket.recv_into`
    2.  `RawIOBase.read_into`
-   These methods avoid creating copies and allocating memory
    -   Received data goes into existing buffer
-   We can convert our program to use `recv_into` and a `memoryview`
    slice to speed up our broadcasting method

In [11]:
class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (4 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)

chunk = write_view[byte_offset : byte_offset + size]
socket.recv_into(chunk)

print(f"New cache: {video_cache=}")

New cache: video_cache=b'\x947^v\xeb\x1ctd\x14\xee\x83f\x1dd\xce\x1flRo/\t7\x9a\xe6\x16\x83Rj\xc5z\xae\xb7\xb0\xefXpG\xce\xc7`\x9cY\xe0\xef\xb4*\x1b\xbe\xaf\xb7\xb2|\x9bF\x85A\xa8\xce\x93rq\xf0,k\x8e7UT\xc8\xf9Y\xe6\x02\xed\xa7U\xdd%I\xec\xa4/\x08\x06\n\xb0\x97)\xb1x\x1d\xea\xdb\xc0\x1a1\xb0\x94\\ 9<\xff\xd1n\x15\xfb\xf4}g\x1a\x1f\xc8%6}\xc6\xa1\x10h\x86%\xdc\x16\x1b{\xb6\xc9\xfc{\xb9;\x9a\xf7\xd5nN\xb6=\xd3\x8a\x07\x9e\xbb\'\x88\xf2FT\x97@m.\x8ep\x10\xa3\x08\x8b\xe83\xf9\xad\x17\xe7\x05\x88\x03\x16O@\x13\x00&S\xb0\x80\xb3\x91\xce+\x11\x93\xa5m%\xa6\xee\xcc\x10\x9bA8\xe5\x83\xa0%5\x04\x00\xee(1\xd6HK*\xdf\xac\xa12\xe3\xa2\xe9*co1\xc5+\xe7\xcf\xc7\xf0\xd6\x04\xff\xa2\x8c\x93:Q\x0b\x9ee\xc8 \x8c\x9e\x7f\xf3\x7f\x10Z\x82l3\x0cbZ\xd0{\x85\x86\xc5\xbe"P:\x00)\xa81dL\x88\x84\x9e\xb5#>\x94;l\xaa\xac\x00\xbf\x8b\xe4\n\x01"\xb4\x93\xde\xee~s\xac\x9b\xb1G\x01\n\x0fM\xeb\xf6\x84\xb3`\xfdB\xd6\xcah\xb8\xca\x95"\x80\xbd4\x8cG\x82+\xb5Y'

-   We can again microbenchmark the result for a $1$ MB chunk

In [12]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (1024 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)


def run_test():
    chunk = write_view[byte_offset : byte_offset + size]
    socket.recv_into(chunk)

result = (
    timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100
)

print(f"{result:0.9f} seconds")

0.000032253 seconds

-   On my machine this takes about $90 \;\mu\text{s}$. Which means we
    could support,

$$
\begin{align}
    \frac{1 \text{ MB}}{90 \; \mu\text{s}} &= 11 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Which also supports,

$$
\begin{align}
    \frac{11 \text{ GB}}{1 \text{MB}} &= 11,000 \text{ processes}
\end{align}
$$

-   Much better scalability

## Things to Remember

-   `memoryview` provides zero-copy methods for reading and writing to
    slices of objects supporting the buffer protocol
-   `bytearray` built-in provides a mutable `bytes`-like type
    -   Can be used for zero-copy data reads
    -   Works with functions like `socket.recv_into`
-   `memoryview` can wrap a `bytearray`
    -   Let’s received data to be spliced into an existing buffer
    -   No need for extra copies